# Retrieval-Augmented Generation Pipeline
The objective of this notebook is to integrate the retrieval component with a Large Language Model (LLM) to create a Retrieval-Augmented Generation (RAG) system.

The retrieval process identifies document chunks that are semantically relevant to a user's question. These chunks are then supplied as context to an LLM, allowing the model to generate answers grounded in university documentation rather than relying solely on its pretrained knowledge.

This notebook represents the first end-to-end implementation of the proposed RAG architecture.

In [26]:
# Install libraries required for the RAG pipeline

!pip install -q pypdf
!pip install -q sentence-transformers
!pip install -q langchain-text-splitters
!pip install -q faiss-cpu
!pip install -q transformers
!pip install -q torch

In [27]:
# Import libraries used throughout the notebook

from pypdf import PdfReader

from langchain_text_splitters import RecursiveCharacterTextSplitter

from sentence_transformers import SentenceTransformer

from transformers import pipeline

import numpy as np

import faiss

In [28]:
# Import libraries required for document loading, chunking,
# embedding generation and vector search

from pypdf import PdfReader

from langchain_text_splitters import RecursiveCharacterTextSplitter

from sentence_transformers import SentenceTransformer

import faiss
import numpy as np


# Load the General Academic Regulations document

pdf_path = "/kaggle/input/datasets/makhosazanemthethwa/cos802-university-documents/General_Academic_Regulations_2026.pdf"

reader = PdfReader(pdf_path)


# Extract text from all pages

full_text = ""

for page in reader.pages:

    page_text = page.extract_text()

    if page_text:
        full_text += page_text + "\n"


# Display document statistics

print(f"Characters: {len(full_text):,}")
print(f"Words: {len(full_text.split()):,}")


# Split the document into overlapping chunks

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_text(full_text)

print(f"Chunks Created: {len(chunks)}")


# Load the Sentence Transformer model

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)


# Generate embeddings for all chunks

embeddings = model.encode(
    chunks,
    show_progress_bar=True
)

embeddings = np.array(embeddings).astype("float32")

print(f"Embeddings Generated: {len(embeddings)}")
print(f"Embedding Dimension: {embeddings.shape[1]}")


# Create a FAISS vector index

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embeddings)

print(f"Vectors Stored: {index.ntotal}")

Characters: 361,882
Words: 53,391
Chunks Created: 455


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (6 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/15 [00:00<?, ?it/s]

Embeddings Generated: 455
Embedding Dimension: 384
Vectors Stored: 455


In [29]:
# Load the FLAN-T5 model and tokenizer

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

llm = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("Model loaded successfully")

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Model loaded successfully


In [30]:
# Retrieve relevant chunks from the FAISS vector store

def retrieve_context(question, top_k=3):

    query_embedding = model.encode([question])

    query_embedding = np.array(query_embedding).astype("float32")

    distances, indices = index.search(
        query_embedding,
        top_k
    )

    retrieved_chunks = [
        chunks[idx]
        for idx in indices[0]
    ]

    return "\n\n".join(retrieved_chunks)


In [55]:
# Define a test question

question = "What is the minimum entry requirement for admission to a master's degree?"

In [56]:
# Retrieve supporting context

context = retrieve_context(question)

print(context[:3000])


33 
G31 Admission 
 
1. Compliance with the minimum requirements will not guarantee admission to a master’s 
degree at the University. In addition to institutional requirements, programme- specific 
admission requirements are determined by the relevant faculty board, approved by Senate 
and included in the relevant faculty yearbook. Furthermore, specific admission criteria may 
be applied, as the University only has a limited number of places available. A candidate may 
also be refused admission to a master’s degree, even if they meet the standard of 
competence in the discipline, if the relevant department lacks adequate supervision capacity. 
 
2. A candidate is admitted to a master’s programme only if they  are in possession of an 
appropriate honours degree, or an equivalent qualification (such as a professional bachelor’s 
degree or a postgraduate diploma) at NQF Level 8 in the same or a relevant field of study.

comprises at least 45 credits, consisting of either a single researc

In [47]:
# Convert the prompt into model inputs

inputs = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True,
    max_length=1024
)

# Generate an answer using FLAN-T5

outputs = llm.generate(
    **inputs,
    max_new_tokens=150
)

# Convert model output into readable text

answer = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(answer)

A bachelor's degree


In [58]:
prompt = f"""
Based only on the context below, answer the question in one sentence.

Question:
{question}

Context:
{context}

Answer:
"""

In [59]:
print("QUESTION:")
print(question)

print("\nRETRIEVED CONTEXT:")
print(context)

QUESTION:
What is the minimum entry requirement for admission to a master's degree?

RETRIEVED CONTEXT:
33 
G31 Admission 
 
1. Compliance with the minimum requirements will not guarantee admission to a master’s 
degree at the University. In addition to institutional requirements, programme- specific 
admission requirements are determined by the relevant faculty board, approved by Senate 
and included in the relevant faculty yearbook. Furthermore, specific admission criteria may 
be applied, as the University only has a limited number of places available. A candidate may 
also be refused admission to a master’s degree, even if they meet the standard of 
competence in the discipline, if the relevant department lacks adequate supervision capacity. 
 
2. A candidate is admitted to a master’s programme only if they  are in possession of an 
appropriate honours degree, or an equivalent qualification (such as a professional bachelor’s 
degree or a postgraduate diploma) at NQF Level 8 in the 

## Observations
The retrieval component successfully identified highly relevant document chunks relating to admission requirements for master's programmes.

However, although the correct information was present in the retrieved context, the language model did not consistently select the most relevant statement when generating an answer. In one experiment, the model incorrectly returned "A bachelor's degree" despite the context explicitly stating that admission to a master's programme requires an appropriate honours degree or an equivalent qualification at NQF Level 8.

This result demonstrates that successful retrieval does not necessarily guarantee accurate answer generation. The quality of the generated response depends on both retrieval effectiveness and the language model's ability to use the retrieved context correctly.

## Conclusion
The first end-to-end Retrieval-Augmented Generation (RAG) pipeline was successfully implemented.

The system was able to retrieve relevant document chunks from the FAISS vector store and provide them as context to a Large Language Model for answer generation.

Testing demonstrated that retrieval quality was generally strong, with relevant regulatory information consistently retrieved. However, answer generation was not always fully aligned with the most relevant retrieved evidence, highlighting an area for further investigation and optimisation.

Future work will focus on prompt refinement, testing alternative language models and evaluating answer quality across a larger set of questions.